# GraphCast: 10-day forecast for Dhaka

Modified from [DeepMind's GraphCast Colab](https://github.com/deepmind/graphcast).

**Goal:** Run GraphCast and get a **10-day forecast at 2 m** for any location. You set **(lat, lon)**; the notebook outputs:
- **1.** Air temperature (°C) · **2.** Relative humidity (%) · **3.** Solar radiation (W/m²) · **4.** Wind speed (m/s) · **5.** WBGT (°C)
- **Excel** file with all forecasted data
- **One plot per variable** (x = time with day and hour, y = value)

**Setup:** Use a **GPU or TPU** runtime in Colab. Set your coordinates in **Section 2** (default: Dhaka 23.81°N, 90.41°E).

## 1. Install and fix environment

In [ ]:
%pip install --upgrade -q https://github.com/deepmind/graphcast/archive/master.zip

# Avoid cartopy/shapely crashes in Colab
!pip uninstall -y shapely 2>/dev/null; !pip install -q shapely --no-binary shapely
!pip install -q openpyxl

In [ ]:
import dataclasses
import functools
import re
from typing import Optional

from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
import haiku as hk
import jax
import matplotlib.pyplot as plt
import numpy as np
import xarray

FORECAST_DAYS = 10
STEPS_PER_DAY = 4
TOTAL_STEPS_10D = FORECAST_DAYS * STEPS_PER_DAY

def parse_file_parts(file_name):
    return dict(part.split("-", 1) for part in file_name.replace(".nc", "").split("_"))

## 2. Location (latitude, longitude)

Set the coordinates for the forecast. Default: Dhaka, Bangladesh.

In [ ]:
LAT = 23.81   # latitude (°N)
LON = 90.41   # longitude (°E)
print(f"Forecast location: {LAT}°N, {LON}°E")

### Model choice

- **`"best"`** – Full GraphCast (0.25° resolution, 37 pressure levels). Best accuracy; needs more RAM/GPU (e.g. Colab Pro or high-memory runtime).
- **`"small"`** – GraphCast_small (1° resolution, 13 levels). Fast and light; good for free Colab.

In [ ]:
MODEL_CHOICE = "small"   # set to "best" for full 0.25° / 37-level model
print("Model choice:", MODEL_CHOICE)

## 3. Connect to GCS and choose model + data

In [ ]:
gcs_client = storage.Client.create_anonymous_client()
gcs_bucket = gcs_client.get_bucket("dm_graphcast")
dir_prefix = "graphcast/"

params_file_options = [b.name.removeprefix(dir_prefix + "params/") for b in gcs_bucket.list_blobs(prefix=dir_prefix + "params/") if b.name.removeprefix(dir_prefix + "params/")]

if MODEL_CHOICE == "best":
    # Full GraphCast: 0.25°, 37 levels (best accuracy; needs more memory)
    preferred = "GraphCast - ERA5 1979-2017 - resolution 0.25 - pressure levels 37 - mesh 2to6 - precipitation input and output.npz"
    fallback = [p for p in params_file_options if "0.25" in p and "37" in p and "small" not in p.lower()]
else:
    # GraphCast_small: 1°, 13 levels (faster, less memory)
    preferred = "GraphCast_small - ERA5 1979-2015 - resolution 1 - pressure levels 13 - mesh 2to5 - precipitation input and output.npz"
    fallback = [p for p in params_file_options if "small" in p.lower() and "13" in p]

params_file = preferred if preferred in params_file_options else (fallback[0] if fallback else params_file_options[0])
dataset_file_options = [b.name.removeprefix(dir_prefix + "dataset/") for b in gcs_bucket.list_blobs(prefix=dir_prefix + "dataset/") if b.name.removeprefix(dir_prefix + "dataset/")]
print("Model params:", params_file)

In [ ]:
with gcs_bucket.blob(f"{dir_prefix}params/{params_file}").open("rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
params = ckpt.params
state = {}
model_config = ckpt.model_config
task_config = ckpt.task_config
print("Model:", ckpt.description[:200], "...")

In [ ]:
dataset_file_options = [b.name.removeprefix(dir_prefix + "dataset/") for b in gcs_bucket.list_blobs(prefix=dir_prefix + "dataset/") if b.name.removeprefix(dir_prefix + "dataset/")]

def steps_from_name(name):
    parts = parse_file_parts(name)
    try:
        return int(parts.get("steps", 0))
    except (ValueError, TypeError):
        return 0

def data_valid_for_model(file_name, model_config, task_config):
    parts = parse_file_parts(file_name.removesuffix(".nc") if file_name.endswith(".nc") else file_name)
    try:
        res = float(parts.get("res", 0))
    except (ValueError, TypeError):
        res = 0
    try:
        levels = int(parts.get("levels", 0))
    except (ValueError, TypeError):
        levels = 0
    precip_ok = (
        ("total_precipitation_6hr" in task_config.input_variables and parts.get("source") in ("era5", "fake")) or
        ("total_precipitation_6hr" not in task_config.input_variables and parts.get("source") in ("hres", "fake"))
    )
    return (
        model_config.resolution in (0, res) and
        len(task_config.pressure_levels) == levels and
        precip_ok
    )

valid = [f for f in dataset_file_options if data_valid_for_model(f, model_config, task_config)]
if not valid:
    valid = dataset_file_options
dataset_file = max(valid, key=steps_from_name)
available_steps = steps_from_name(dataset_file)
eval_steps = min(TOTAL_STEPS_10D, max(1, available_steps - 2))
print("Dataset:", dataset_file)
print("Available time steps in file:", available_steps)
print("Forecast steps we will run:", eval_steps, "(", eval_steps * 6 / 24, "days)")

## 5. Load data

In [ ]:
with gcs_bucket.blob(f"{dir_prefix}dataset/{dataset_file}").open("rb") as f:
    example_batch = xarray.load_dataset(f).compute()

assert example_batch.dims["time"] >= 2 + eval_steps, "Dataset has too few steps for requested forecast length."
print("Data dims:", example_batch.dims)

In [ ]:
eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch,
    target_lead_times=slice("6h", f"{eval_steps * 6}h"),
    **dataclasses.asdict(task_config),
)
print("Eval inputs:", eval_inputs.dims)
print("Eval targets (forecast steps):", eval_targets.dims)

### Preview preliminary input data (at your location)

Initial state fed into the model: **2 time steps** at (LAT, LON). The forecast starts from the state after these inputs.

In [ ]:
import pandas as pd

lat_dim = "lat" if "lat" in eval_inputs.coords else "latitude"
lon_dim = "lon" if "lon" in eval_inputs.coords else "longitude"
input_at_point = eval_inputs.sel({lat_dim: LAT, lon_dim: LON}, method="nearest")

rows = []
for i in range(eval_inputs.sizes["time"]):
    row = {"input_step": i}
    t = eval_inputs.time.isel(time=i).values
    try:
        row["time"] = str(pd.Timestamp(t))
    except Exception:
        row["time"] = str(t)
    for v in ["2m_temperature", "mean_sea_level_pressure", "10m_u_component_of_wind", "10m_v_component_of_wind", "total_precipitation_6hr"]:
        if v in input_at_point:
            val = float(input_at_point[v].isel(time=i).squeeze().values)
            if v == "2m_temperature" and val > 200:
                val = round(val - 273.15, 2)
                row[v] = f"{val} °C"
            elif v == "mean_sea_level_pressure":
                row[v] = round(val, 2)
            else:
                row[v] = round(val, 4)
    rows.append(row)

input_df = pd.DataFrame(rows)
print("Preliminary input data at", LAT, "°N,", LON, "°E (initial conditions for the forecast):")
display(input_df)
print("\nRaw example_batch time range (first few steps):", example_batch.time.isel(time=slice(0, min(4, example_batch.sizes["time"]))).values)

In [ ]:
with gcs_bucket.blob(dir_prefix + "stats/diffs_stddev_by_level.nc").open("rb") as f:
    diffs_stddev_by_level = xarray.load_dataset(f).compute()
with gcs_bucket.blob(dir_prefix + "stats/mean_by_level.nc").open("rb") as f:
    mean_by_level = xarray.load_dataset(f).compute()
with gcs_bucket.blob(dir_prefix + "stats/stddev_by_level.nc").open("rb") as f:
    stddev_by_level = xarray.load_dataset(f).compute()

## 6. Build predictor and run 10-day rollout

In [ ]:
def construct_wrapped_graphcast(mconfig, tconfig):
    predictor = graphcast.GraphCast(mconfig, tconfig)
    predictor = casting.Bfloat16Cast(predictor)
    predictor = normalization.InputsAndResiduals(
        predictor,
        diffs_stddev_by_level=diffs_stddev_by_level,
        mean_by_level=mean_by_level,
        stddev_by_level=stddev_by_level,
    )
    return autoregressive.Predictor(predictor, gradient_checkpointing=True)


@hk.transform_with_state
def run_forward(mconfig, tconfig, inputs, targets_template, forcings):
    predictor = construct_wrapped_graphcast(mconfig, tconfig)
    return predictor(inputs, targets_template=targets_template, forcings=forcings)


def with_configs(fn):
    return functools.partial(fn, mconfig=model_config, tconfig=task_config)


def with_params(fn):
    return functools.partial(fn, params=params, state=state)
 

def drop_state(fn):
    return lambda **kw: fn(**kw)[0]


run_forward_jitted = drop_state(with_params(jax.jit(with_configs(run_forward.apply))))

In [ ]:
print("Running autoregressive rollout (", eval_steps, "steps = ", eval_steps * 6 / 24, "days )...")
predictions = rollout.chunked_prediction(
    run_forward_jitted,
    rng=jax.random.PRNGKey(0),
    inputs=eval_inputs,
    targets_template=eval_targets * np.nan,
    forcings=eval_forcings,
)
print("Done. Predictions dims:", predictions.dims)

## 7. Extract at (LAT, LON): 5 variables, Excel export, and plots

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

lat_dim = "lat" if "lat" in predictions.coords else "latitude"
lon_dim = "lon" if "lon" in predictions.coords else "longitude"
point = predictions.sel({lat_dim: LAT, lon_dim: LON}, method="nearest")

# Reference time: first forecast step is 6h after last input
t0 = example_batch.time.isel(time=0).values
try:
    base = pd.Timestamp(t0) + pd.Timedelta(hours=6)
except Exception:
    base = pd.Timestamp("2000-01-01") + pd.Timedelta(hours=6)
times = [base + pd.Timedelta(hours=6*i) for i in range(eval_steps)]

# 1. Air temperature at 2m (°C)
t2m_k = point["2m_temperature"].squeeze().values if "2m_temperature" in point else np.full(eval_steps, np.nan)
air_temperature = (t2m_k - 273.15) if np.nanmin(t2m_k) > 200 else t2m_k

# 2. Relative humidity (%): from specific_humidity (lowest level) + 2m T + pressure
if "specific_humidity" in point and "level" in point["specific_humidity"].dims:
    q_surf = point["specific_humidity"].isel(level=-1).squeeze().values  # highest pressure = near surface
else:
    q_surf = np.full(eval_steps, np.nan)
p_surf = point["mean_sea_level_pressure"].squeeze().values if "mean_sea_level_pressure" in point else 101325.0
p_surf = np.broadcast_to(np.atleast_1d(p_surf).flat[0] if np.isscalar(p_surf) else p_surf, eval_steps)
e_sat = 611.2 * np.exp(17.67 * air_temperature / (air_temperature + 243.5))
e = np.clip(q_surf * p_surf / (0.622 + 0.378 * np.clip(q_surf, 1e-10, 1)), 1e-10, None)
relative_humidity = np.clip(100 * e / (e_sat + 1e-10), 0, 100)
relative_humidity = np.where(np.isnan(q_surf), np.nan, relative_humidity)

# 3. Solar radiation (TOA, W/m²) from forcings
forc_pt = eval_forcings.sel({lat_dim: LAT, lon_dim: LON}, method="nearest")
if "toa_incident_solar_radiation" in forc_pt:
    solar_radiation = forc_pt["toa_incident_solar_radiation"].squeeze().values
else:
    solar_radiation = np.full(eval_steps, np.nan)
if np.size(solar_radiation) != eval_steps:
    solar_radiation = np.broadcast_to(np.nanmean(solar_radiation) if np.size(solar_radiation) else np.nan, eval_steps)

# 4. Wind speed (m/s)
u10 = point["10m_u_component_of_wind"].squeeze().values if "10m_u_component_of_wind" in point else np.zeros(eval_steps)
v10 = point["10m_v_component_of_wind"].squeeze().values if "10m_v_component_of_wind" in point else np.zeros(eval_steps)
wind_speed = np.sqrt(np.asarray(u10)**2 + np.asarray(v10)**2)

# 5. WBGT (°C): simplified from T and RH (Stull wet-bulb then WBGT ≈ 0.7*Tw + 0.3*T)
def wet_bulb_stull(T_C, RH):
    a = np.arctan(0.151977 * (RH + 8.313659)**0.5)
    b = np.arctan(T_C + RH) - np.arctan(RH - 1.676331)
    c = 0.00391838 * (RH**1.5) * np.arctan(0.023101 * RH) - 4.686035
    return T_C * a + b + c
Tw = wet_bulb_stull(air_temperature, relative_humidity)
WBGT = 0.7 * Tw + 0.3 * air_temperature
WBGT = np.where(np.isnan(air_temperature) | np.isnan(relative_humidity), np.nan, WBGT)

# DataFrame and Excel
df = pd.DataFrame({
    "datetime": times,
    "day": [t.strftime("%Y-%m-%d") for t in times],
    "hour": [t.hour for t in times],
    "air_temperature_2m_C": np.round(air_temperature, 2),
    "relative_humidity_pct": np.round(relative_humidity, 2),
    "solar_radiation_Wm2": np.round(solar_radiation, 2),
    "wind_speed_ms": np.round(wind_speed, 2),
    "WBGT_C": np.round(WBGT, 2),
})
excel_path = f"graphcast_forecast_{LAT}_{LON}.xlsx"
df.to_excel(excel_path, index=False)
print("Saved:", excel_path)
df

In [ ]:
# One plot per variable: x = time (day and hour), y = value
x_labels = [t.strftime("%m-%d %H:%M") for t in times]
x_ix = np.arange(len(times))

fig, axes = plt.subplots(5, 1, figsize=(12, 12), sharex=True)
fig.suptitle(f"10-day forecast at {LAT}°N, {LON}°E (2 m)", fontsize=14)

axes[0].plot(x_ix, df["air_temperature_2m_C"], "o-", color="C0", markersize=4)
axes[0].set_ylabel("Air temperature (°C)")
axes[0].set_title("1. Air temperature (2 m)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_ix, df["relative_humidity_pct"], "o-", color="C1", markersize=4)
axes[1].set_ylabel("Relative humidity (%)")
axes[1].set_title("2. Relative humidity")
axes[1].grid(True, alpha=0.3)

axes[2].plot(x_ix, df["solar_radiation_Wm2"], "o-", color="C2", markersize=4)
axes[2].set_ylabel("Solar radiation (W/m²)")
axes[2].set_title("3. Solar radiation (TOA)")
axes[2].grid(True, alpha=0.3)

axes[3].plot(x_ix, df["wind_speed_ms"], "o-", color="C3", markersize=4)
axes[3].set_ylabel("Wind speed (m/s)")
axes[3].set_title("4. Wind speed (10 m)")
axes[3].grid(True, alpha=0.3)

axes[4].plot(x_ix, df["WBGT_C"], "o-", color="C4", markersize=4)
axes[4].set_ylabel("WBGT (°C)")
axes[4].set_xlabel("Time (day and hour)")
axes[4].set_title("5. WBGT (Wet Bulb Globe Temperature)")
axes[4].grid(True, alpha=0.3)

plt.xticks(x_ix[::max(1, len(x_ix)//12)], [x_labels[i] for i in range(0, len(x_labels), max(1, len(x_labels)//12))], rotation=45, ha="right")
plt.tight_layout()
plt.show()